# Notebook for finding the parameters where recall is 1 and runtime is good

In [ ]:
import os
import sys
import itertools


def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from utils.helpers.measure_similarities import *

## Read Bucketing Runtime CSV

In [ ]:
CITY = "rome"
BUCKETING_METHOD = "loose"
MEASURE = "dtw"
DATA_SIZE = [500]
SCHEME = "grid"


folder_path = f"../../results_hashed/runtimes/bucketing/{CITY}/true_trajectories/{BUCKETING_METHOD}/{MEASURE}/"
file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(true_trajectories)_{CITY}_{MEASURE}_{DATA_SIZE}_{SCHEME}.csv"
file_path = folder_path + file_name

runtime_bucketing = pd.read_csv(file_path, index_col=0)
runtime_bucketing.head()



runtime_bucketing.head(5)

## Read No Bucketing Runtime CSV

In [ ]:
CITY = "rome"
MEASURE = "dtw"
DATA_SIZE = [500]
SCHEME = "grid"


folder_path = f"../../results_hashed/runtimes/no_bucketing/{CITY}/{MEASURE}/"
file_name = f"runtimes_{CITY}_{MEASURE}_{DATA_SIZE}_{SCHEME}_no_bucketing.csv"
file_path = folder_path + file_name

runtime_no_bucketing = pd.read_csv(file_path)
runtime_no_bucketing.head(5)


## Read No Bucketing Correlation

In [ ]:

folder_path = f"../correlation/no_bucketing_correlation_values/"
file_name = f"correlation_results_{CITY}_grid_no_bucketing.csv"
file_path = folder_path + file_name

correlation_no_bucketing = pd.read_csv(file_path, index_col=0)
correlation_no_bucketing.head(5)

#Merge runtime_no_bucketing and correlation_no_bucketing on columns "Resolution" and "Layers"
runtime_no_bucketing = pd.merge(runtime_no_bucketing, correlation_no_bucketing, on=["Resolution", "Layers"])
runtime_no_bucketing.head(5)


## Read Bucket Evaluation CSV

In [ ]:
CITY = "rome"
BUCKETING_METHOD = "loose"
DATA_SIZE = 500
SCHEME = "grid"

folder_path = f"../../results_hashed/bucket_evaluation/{BUCKETING_METHOD}/"
file_name = f"{CITY}_{MEASURE}_{SCHEME}_{DATA_SIZE}_{BUCKETING_METHOD}.csv"
file_path = folder_path + file_name

bucket_evaluation = pd.read_csv(file_path, index_col=0)

bucket_evaluation.head(5)

In [ ]:
# First, filter the bucket_evaluation dataframe
filtered_eval = bucket_evaluation[
    (bucket_evaluation["Avg Recall"] == 1) &
    (bucket_evaluation["Threshold"] == 0.1)
]

# Select only the necessary columns from runtime_bucketing
runtime_selected = runtime_bucketing[[
    "Resolution", "Layers",
    "Average Similarity Computation Time (Seconds)",
    "Average Hash Generation Time (Seconds)",
    "Average Bucket Distribution Time (Seconds)",
    "Total time (Seconds)"
]]

# Merge using only Diameter, Layers, and Disks as keys
runtime_bucketing_and_bucket_evaluation = filtered_eval.merge(
    runtime_selected,
    on=["Resolution", "Layers"],
    how="left"
)

columns_to_show = [
    "Resolution", "Layers",
    "Avg Correlation Coefficient",
    "Avg Recall", "Threshold",
    "Average Similarity Computation Time (Seconds)",
    "Average Hash Generation Time (Seconds)",
    "Average Bucket Distribution Time (Seconds)",
    "Total time (Seconds)"
]

runtime_bucketing_and_bucket_evaluation.head(5)



In [ ]:
# Sort the merged dataframe by Avg Correlation Coefficient in descending order
runtime_bucketing_and_bucket_evaluation_sorted = runtime_bucketing_and_bucket_evaluation.sort_values(
    by="Total time (Seconds)",
    ascending=True
)
runtime_bucketing_and_bucket_evaluation_sorted.head(10)

#show only these columns
columns_to_show = [
    "Resolution", "Layers",
    "Avg Correlation Coefficient",
    "Avg Recall", "Threshold",
    "Average Similarity Computation Time (Seconds)",
    "Average Hash Generation Time (Seconds)",
    "Average Bucket Distribution Time (Seconds)",
    "Total time (Seconds)"
]
runtime_bucketing_and_bucket_evaluation_sorted = runtime_bucketing_and_bucket_evaluation_sorted[columns_to_show]

#extract lowest runtime Total time (Seconds) in the runtime_bucketing_and_bucket_evaluation_sorted dataframe
lowest_runtime = runtime_bucketing_and_bucket_evaluation_sorted.iloc[0]

lowest_runtime = runtime_bucketing_and_bucket_evaluation_sorted["Total time (Seconds)"].min()


print(lowest_runtime)


runtime_bucketing_and_bucket_evaluation_sorted.head(5)



In [ ]:
runtime_no_bucketing.head(5)


# Merge runtime_no_bucketing with merged_df_sorted top 5 rows
runtime_no_bucketing_vs_runtime_bucketing = runtime_bucketing_and_bucket_evaluation_sorted.head(5)
runtime_no_bucketing_vs_runtime_bucketing = runtime_no_bucketing_vs_runtime_bucketing.merge(
    runtime_no_bucketing,
    on=["Resolution", "Layers"],
    how="left",
    suffixes=("", "_no_bucketing")
)
# Rename the columns to avoid confusion
runtime_no_bucketing_vs_runtime_bucketing.rename(
    columns={
        "Average Similarity Computation Time (Seconds)": "Average Similarity Computation Time (Seconds)_bucketing",
        "Average Hash Generation Time (Seconds)": "Average Hash Generation Time (Seconds)_bucketing",
        "Average Bucket Distribution Time (Seconds)": "Average Bucket Distribution Time (Seconds)_bucketing",
        "Total time (Seconds)": "Total time (Seconds)_bucketing"
    },
    inplace=True
)




runtime_no_bucketing_vs_runtime_bucketing.head()


In [ ]:
# Filter the runtime_no_bucketing dataframe to only include the rows where Layers = 3
filtered_runtime_no_bucketing = runtime_no_bucketing[runtime_no_bucketing["Layers"] == 3]

filtered_runtime_no_bucketing.head(5)


import matplotlib.pyplot as plt
import numpy as np

def plot_total_time_vs_resolution(
    df,
    city: str,
    measure: str = "dtw",
    layers: int = 3,
    size: int = 500,
    lowest_runtime: float = None,  # Add this argument
):
    """
    Plots the relationship between grid resolution and total time (in seconds),
    and overlays correlation on a secondary y-axis with clear labels.
    Adds a horizontal line for the externally provided lowest total runtime.
    """

    # Filter the DataFrame
    filtered = df[
        (df['City'] == city) &
        (df['Measure'] == measure) &
        (df['Layers'] == layers) &
        (df['Size'] == size)
    ].sort_values(by='Resolution')

    # Extract values
    res = filtered['Resolution'].values
    total_time = filtered['Total time (Seconds)'].values
    corr = filtered['Correlation'].astype(float).values

    # Plot
    fig, ax1 = plt.subplots(figsize=(10, 8), dpi=100)
    ax2 = ax1.twinx()
    cmap = plt.get_cmap("gist_ncar")

    line1 = ax1.plot(res, total_time, color=cmap(0.3), marker='o', lw=2, label='Total Time No Bucketing')[0]
    line2 = ax2.plot(res, corr, color=cmap(0.7), marker='o', lw=2, linestyle='--', label='Pearson correlation coefficient')[0]

    # Use provided lowest_runtime if available
    if lowest_runtime is not None:
        line3 = ax1.axhline(
            y=lowest_runtime,
            color='red',
            linestyle=':',
            linewidth=2,
            label=f'Lowest Runtime Bucketing = {lowest_runtime:.3f}s'
        )
        lines = [line1, line2, line3]
    else:
        lines = [line1, line2]

    labels = [line.get_label() for line in lines]
    ax1.legend(lines, labels, loc="upper right", fontsize=14)

    # Axis labels and styling
    ax1.set_xlabel("Grid resolution (km)", fontsize=18)
    ax1.set_ylabel("Total Time (Seconds)", fontsize=18)
    ax2.set_ylabel("Correlation", fontsize=18)
    

    ax1.tick_params(axis="both", which="major", labelsize=16)
    ax2.tick_params(axis="both", which="major", labelsize=16)

    ax1.set_title(
        f"{city.capitalize()}: Total Time & Correlation vs Resolution\nLayers: {layers} | Size: {size}",
        fontsize=14,
        color='grey'
    )

    ax1.grid(True)
    fig.tight_layout()
    plt.show()




plot_total_time_vs_resolution(
    filtered_runtime_no_bucketing,
    city=CITY,
    measure="dtw",
    layers=3,
    size=500,
    lowest_runtime=lowest_runtime  # Pass the value from your earlier cell
)


# filtered_runtime_no_bucketing.head(5)
